# 纯torch代码实现默认参数下的Embedding全过程


In [1]:
import json

import torch
# 加载权重文件
from safetensors.torch import load_file
from torch import nn
import torch.nn.functional as F


In [2]:
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

texts = ["Hello Word, a test sentence"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 从前一步得到的结果
token_ids = [9707, 9322, 11, 264, 1273, 11652, 151643]

vocab_size = 151669
# 查询得知

hidden_size = 1024
padding_idx = 151643

In [3]:
# 加载配置文件
config_file = "/Users/junzerg/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/config.json"

with open(config_file, "r", encoding="utf-8") as f:
    config = json.load(f)
config

{'architectures': ['Qwen3ForCausalLM'],
 'attention_bias': False,
 'attention_dropout': 0.0,
 'bos_token_id': 151643,
 'eos_token_id': 151643,
 'head_dim': 128,
 'hidden_act': 'silu',
 'hidden_size': 1024,
 'initializer_range': 0.02,
 'intermediate_size': 3072,
 'max_position_embeddings': 32768,
 'max_window_layers': 28,
 'model_type': 'qwen3',
 'num_attention_heads': 16,
 'num_hidden_layers': 28,
 'num_key_value_heads': 8,
 'rms_norm_eps': 1e-06,
 'rope_scaling': None,
 'rope_theta': 1000000,
 'sliding_window': None,
 'tie_word_embeddings': True,
 'torch_dtype': 'bfloat16',
 'transformers_version': '4.51.3',
 'use_cache': True,
 'use_sliding_window': False,
 'vocab_size': 151669}

# 2. Embedding lookup
根据 token id 在 embedding 矩阵中索引对应向量

In [4]:
# nn.Embedding
embed_tokens = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=hidden_size,
    padding_idx=padding_idx,
)
embed_tokens

Embedding(151669, 1024, padding_idx=151643)

In [5]:
token_ids = torch.tensor(token_ids, dtype=torch.long)
token_ids


tensor([  9707,   9322,     11,    264,   1273,  11652, 151643])

# 模型权重加载
后面就需要用到模型权重了。这里模型权重处理

In [6]:
model_file = "/Users/junzerg/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/model.safetensors"

# 读取文件
state_dict = load_file(model_file, device="cpu")  # 返回 dict: key -> torch.Tensor

# 查看有哪些 key
print(list(state_dict.keys()))

['embed_tokens.weight', 'layers.0.input_layernorm.weight', 'layers.0.mlp.down_proj.weight', 'layers.0.mlp.gate_proj.weight', 'layers.0.mlp.up_proj.weight', 'layers.0.post_attention_layernorm.weight', 'layers.0.self_attn.k_norm.weight', 'layers.0.self_attn.k_proj.weight', 'layers.0.self_attn.o_proj.weight', 'layers.0.self_attn.q_norm.weight', 'layers.0.self_attn.q_proj.weight', 'layers.0.self_attn.v_proj.weight', 'layers.1.input_layernorm.weight', 'layers.1.mlp.down_proj.weight', 'layers.1.mlp.gate_proj.weight', 'layers.1.mlp.up_proj.weight', 'layers.1.post_attention_layernorm.weight', 'layers.1.self_attn.k_norm.weight', 'layers.1.self_attn.k_proj.weight', 'layers.1.self_attn.o_proj.weight', 'layers.1.self_attn.q_norm.weight', 'layers.1.self_attn.q_proj.weight', 'layers.1.self_attn.v_proj.weight', 'layers.10.input_layernorm.weight', 'layers.10.mlp.down_proj.weight', 'layers.10.mlp.gate_proj.weight', 'layers.10.mlp.up_proj.weight', 'layers.10.post_attention_layernorm.weight', 'layers.10.

In [7]:
# 大致看起来没有异常的键，但还是整理一下
# 处理键名（如果需要）
# 例如：移除 "model." 前缀
new_state_dict = {}
for key, value in state_dict.items():
    new_key = key
    # 根据你的模型结构调整
    # new_key = key.replace("model.", "")
    new_state_dict[new_key] = value
print(list(new_state_dict.keys()))

['embed_tokens.weight', 'layers.0.input_layernorm.weight', 'layers.0.mlp.down_proj.weight', 'layers.0.mlp.gate_proj.weight', 'layers.0.mlp.up_proj.weight', 'layers.0.post_attention_layernorm.weight', 'layers.0.self_attn.k_norm.weight', 'layers.0.self_attn.k_proj.weight', 'layers.0.self_attn.o_proj.weight', 'layers.0.self_attn.q_norm.weight', 'layers.0.self_attn.q_proj.weight', 'layers.0.self_attn.v_proj.weight', 'layers.1.input_layernorm.weight', 'layers.1.mlp.down_proj.weight', 'layers.1.mlp.gate_proj.weight', 'layers.1.mlp.up_proj.weight', 'layers.1.post_attention_layernorm.weight', 'layers.1.self_attn.k_norm.weight', 'layers.1.self_attn.k_proj.weight', 'layers.1.self_attn.o_proj.weight', 'layers.1.self_attn.q_norm.weight', 'layers.1.self_attn.q_proj.weight', 'layers.1.self_attn.v_proj.weight', 'layers.10.input_layernorm.weight', 'layers.10.mlp.down_proj.weight', 'layers.10.mlp.gate_proj.weight', 'layers.10.mlp.up_proj.weight', 'layers.10.post_attention_layernorm.weight', 'layers.10.

In [8]:
# 提取 embedding 权重
embedding_weights = state_dict['embed_tokens.weight']
embedding_weights.shape

torch.Size([151669, 1024])

In [9]:
# 直接加载没有问题
embed_tokens.weight.data.copy_(embedding_weights)

tensor([[-0.0031,  0.0327, -0.0703,  ...,  0.0138, -0.0144,  0.0128],
        [ 0.0303,  0.0244, -0.0613,  ..., -0.0031, -0.0374,  0.0077],
        [ 0.0292,  0.0322, -0.0223,  ..., -0.0095,  0.0025,  0.0256],
        ...,
        [ 0.0031, -0.1064,  0.0245,  ...,  0.0028, -0.0055, -0.0026],
        [ 0.0031, -0.1064,  0.0245,  ...,  0.0028, -0.0055, -0.0026],
        [ 0.0031, -0.1064,  0.0245,  ...,  0.0028, -0.0055, -0.0026]])

In [10]:
# 使用 nn.Embedding 查表
inputs_embeds = embed_tokens(token_ids)  # shape: [7, 1024]
print(inputs_embeds.shape)
print(inputs_embeds[0])

torch.Size([7, 1024])
tensor([ 0.0027, -0.0106,  0.0182,  ..., -0.0143, -0.0408,  0.0062],
       grad_fn=<SelectBackward0>)



# 位置信息（Positional Encoding）——告诉模型“顺序”

In [11]:
from transformers import DynamicCache

# kv缓存用的
past_key_values = DynamicCache()
print(f"past_key_values: {past_key_values}")

# past_seen_tokens 表示已处理过的 token 数量，用于计算当前输入在序列中的绝对位置。
past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
print(f"past_seen_tokens: {past_seen_tokens}")

# cache_position 表示当前输入在 KV 缓存中的位置索引，用于增量生成。
# 首次前向：past_seen_tokens = 0，cache_position = [0, 1, 2, ...]
# 后续生成：past_seen_tokens = 已生成长度，cache_position = [已生成长度, 已生成长度+1, ...]
cache_position = torch.arange(
    past_seen_tokens, past_seen_tokens + inputs_embeds.shape[0], device=inputs_embeds.device
)
print(f"cache_position: {cache_position}")

# position_ids 是每个 token 的绝对位置索引，用于位置编码（如 RoPE）。
# 形状：[1, seq_len]
# 内容：[[0, 1, 2, ...]] 或 [[past_seen_tokens, past_seen_tokens+1, ...]]
# position_ids = cache_position.unsqueeze(-1)
# 这里不考虑batch，也就不unsqueeze了
position_ids = cache_position
print(f"position_ids: {position_ids}")

# 计算 attention_mask
# 到这里padding_size是None
attention_mask = torch.ones_like(token_ids)
print(f"attention_mask: {attention_mask}")

/Users/junzerg/Projects/fork/transformers/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


past_key_values: DynamicCache(layers=[])
past_seen_tokens: 0
cache_position: tensor([0, 1, 2, 3, 4, 5, 6])
position_ids: tensor([0, 1, 2, 3, 4, 5, 6])
attention_mask: tensor([1, 1, 1, 1, 1, 1, 1])


# 构建推理模型

In [12]:
# 初始的隐藏层就是上面Embedding lookup的结果
hidden_states = inputs_embeds
print(hidden_states[0])

tensor([ 0.0027, -0.0106,  0.0182,  ..., -0.0143, -0.0408,  0.0062],
       grad_fn=<SelectBackward0>)


## 定义RoPE
旋转位置编码（Rotary Position Embedding, RoPE），这是一种将位置信息编码到 Transformer 模型中的方法。
与传统的绝对位置编码不同，RoPE 通过旋转操作将相对位置信息直接嵌入到 query 和 key 向量中。
论文地址：https://arxiv.org/abs/2104.09864

注意和transformer原始论文Attention is all you need中固定正弦/余弦位置编码（sinusoidal positional encoding, Sine-PE）区别和优点

In [13]:
# 定义RoPE
from rotary_embedding import RotaryEmbedding
rotary_emb = RotaryEmbedding()
position_embeddings = rotary_emb(hidden_states, position_ids=position_ids)
print(f"position_embeddings: {position_embeddings[0][1]}")

position_embeddings: tensor([0.5403, 0.6925, 0.7965, 0.8662, 0.9124, 0.9428, 0.9627, 0.9758, 0.9842,
        0.9897, 0.9933, 0.9957, 0.9972, 0.9982, 0.9988, 0.9992, 0.9995, 0.9997,
        0.9998, 0.9999, 0.9999, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 0.5403, 0.6925, 0.7965, 0.8662, 0.9124, 0.9428, 0.9627, 0.9758,
        0.9842, 0.9897, 0.9933, 0.9957, 0.9972, 0.9982, 0.9988, 0.9992, 0.9995,
        0.9997, 0.9998, 0.9999, 0.9999, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0

## 开始逐层构建推理框架
### Layer层构成


### RMSNorm
RMSNorm（Root Mean Square Layer Normalization） 是一种归一化技术，相当于简化版的 LayerNorm。

他的前向计算步骤：
1. 计算输入的平方的均值（方差）
2. 用均方根的倒数来缩放输入
3. 乘以可学习的权重参数

在 Attention 中使用 RMSNorm 的好处
- 稳定注意力分数的计算：对 Q 和 K 进行归一化后，它们的点积（注意力分数）会更加稳定，防止数值过大或过小
- 提高训练稳定性：避免梯度爆炸或消失，特别是在深层模型中
- 计算效率更高：相比标准 LayerNorm，RMSNorm 不需要计算和减去均值，只需要计算均方根，运算更简单
- 适配大模型训练：在大规模语言模型（如 Qwen3）中，这种设计已被证明能提升性能和稳定性
- 改善注意力质量：归一化后的 Q 和 K 有助于注意力机制更好地捕捉相关性，而不受向量幅度的影响
- 这是现代 Transformer 架构（如 LLaMA、Qwen 等）的一个重要改进，相比原始的 Transformer 设计更加高效和稳定。



In [14]:
from tests.learn_embedding.decode_layer import DecoderLayer
from tests.learn_embedding.rms_norm import RMSNorm
from tests.learn_embedding.rotary_embedding import RotaryEmbedding
# 根据模型配置构建transformer深度神经网络
layers = nn.ModuleList(
    [DecoderLayer(layer_idx) for layer_idx in range(config["num_hidden_layers"])]
)
norm = RMSNorm(
    config["hidden_size"],
    eps=config["rms_norm_eps"],
)
rotary_emb = RotaryEmbedding()

In [15]:
print(layers)
print(norm)
print(rotary_emb)

ModuleList(
  (0-27): 28 x DecoderLayer(
    (self_attn): Attention(
      (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
      (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
      (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
      (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
      (q_norm): RMSNorm((128,), eps=1e-06)
      (k_norm): RMSNorm((128,), eps=1e-06)
    )
    (mlp): MLP(
      (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
      (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
      (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
      (act_fn): SiLU()
    )
    (input_layernorm): RMSNorm((1024,), eps=1e-06)
    (post_attention_layernorm): RMSNorm((1024,), eps=1e-06)
  )
)
RMSNorm((1024,), eps=1e-06)
RotaryEmbedding()


In [16]:
from embedding_model import EmbeddingModel
model = EmbeddingModel(
    embed_tokens=embed_tokens,
    layers=layers,
    norm=norm,
   rotary_emb=rotary_emb,
)
missing, unexpected = model.load_state_dict(state_dict,strict=True)
print("==> 加载完成")
if missing:
    print("missing keys:", missing[:20])
if unexpected:
    print("unexpected keys:", unexpected[:20])


==> 加载完成


# 推理！推理！推理！

In [17]:
print(hidden_states)

tensor([[ 0.0027, -0.0106,  0.0182,  ..., -0.0143, -0.0408,  0.0062],
        [ 0.0031,  0.0045, -0.0206,  ..., -0.0312, -0.0231,  0.0123],
        [-0.0201,  0.0503, -0.0757,  ..., -0.0210,  0.0055, -0.0255],
        ...,
        [-0.0214, -0.0391, -0.0535,  ...,  0.0430, -0.0386, -0.0222],
        [ 0.0170, -0.0674,  0.0131,  ...,  0.0086,  0.0210, -0.0094],
        [-0.0043,  0.0435, -0.0334,  ..., -0.0039,  0.0449,  0.0320]],
       grad_fn=<EmbeddingBackward0>)


In [18]:
for layer in layers:
    hidden_states = layer(hidden_states, position_embeddings=position_embeddings)
    print(hidden_states)
hidden_states

tensor([[-0.1522,  0.2947,  0.1288,  ..., -0.2905,  0.1957,  0.0838],
        [-0.2053,  0.5473, -0.1405,  ...,  0.1150, -0.1174,  0.2250],
        [-0.0407, -0.2456,  0.0588,  ..., -0.2477,  0.2038,  0.1428],
        ...,
        [-0.1980,  0.4296,  0.2834,  ..., -0.0549, -0.0119,  0.2389],
        [-0.3408,  0.9105,  0.2030,  ..., -0.0593,  0.0250,  0.1598],
        [ 0.1197,  0.0706, -0.0216,  ...,  0.0840, -0.2228,  0.0867]],
       grad_fn=<AddBackward0>)
tensor([[ 0.4563,  0.0623, -0.0033,  ..., -0.2317, -0.3280,  0.1696],
        [-0.4150,  1.0764, -0.2604,  ..., -0.0435, -0.4304,  0.0068],
        [-0.1700,  0.1030,  0.0533,  ..., -0.0382, -0.0978,  0.1120],
        ...,
        [-0.3932,  0.7941,  0.2731,  ..., -0.0965, -0.0673,  0.3997],
        [-0.7954,  1.4802,  0.3617,  ...,  0.0665, -0.0586,  0.2672],
        [-0.0707,  0.5347, -0.0643,  ...,  0.0697, -0.3211,  0.1182]],
       grad_fn=<AddBackward0>)
tensor([[ 1.9807e+01,  3.5568e+02,  5.8537e-01,  ..., -1.3404e+00,
   

tensor([[ 3.1839e+01, -2.7776e+01,  1.9448e+01,  ..., -9.5097e+01,
         -1.4699e+02,  1.3067e+01],
        [ 1.6462e+01, -1.1589e+00,  1.5081e+02,  ..., -3.1190e+00,
         -4.0043e+00,  6.5594e+00],
        [ 1.4089e-01, -4.6580e+00,  1.4230e+02,  ..., -1.2459e+01,
         -1.9776e+00, -4.4816e-01],
        ...,
        [ 1.2280e+01, -9.1084e+00,  1.4335e+02,  ..., -2.1508e+01,
         -7.1591e-01,  8.5341e-01],
        [-2.7463e+00, -8.5582e+00,  7.6599e+01,  ...,  2.8469e+00,
          2.0868e+00, -1.9799e+00],
        [-2.1710e+00, -4.4721e+00,  1.0353e+02,  ..., -4.7017e+00,
         -4.4223e+00, -6.3176e+00]], grad_fn=<AddBackward0>)

In [19]:
# 正规化
hidden_states = norm(hidden_states)
print(hidden_states)

tensor([[  2.9908, -10.1464,  -0.0555,  ...,  -7.9758, -12.9312,   1.1300],
        [  4.2988,  -1.1769,  -1.1964,  ...,  -0.7272,  -0.9793,   1.5769],
        [  0.0356,  -4.5765,  -1.0923,  ...,  -2.8105,  -0.4679,  -0.1042],
        ...,
        [  3.3846,  -9.7629,  -1.2004,  ...,  -5.2929,  -0.1848,   0.2166],
        [ -0.9304, -11.2751,  -0.7884,  ...,   0.8611,   0.6621,  -0.6175],
        [ -0.6450,  -5.1673,  -0.9345,  ...,  -1.2473,  -1.2305,  -1.7281]],
       grad_fn=<MulBackward0>)


In [20]:
print(hidden_states)

tensor([[  2.9908, -10.1464,  -0.0555,  ...,  -7.9758, -12.9312,   1.1300],
        [  4.2988,  -1.1769,  -1.1964,  ...,  -0.7272,  -0.9793,   1.5769],
        [  0.0356,  -4.5765,  -1.0923,  ...,  -2.8105,  -0.4679,  -0.1042],
        ...,
        [  3.3846,  -9.7629,  -1.2004,  ...,  -5.2929,  -0.1848,   0.2166],
        [ -0.9304, -11.2751,  -0.7884,  ...,   0.8611,   0.6621,  -0.6175],
        [ -0.6450,  -5.1673,  -0.9345,  ...,  -1.2473,  -1.2305,  -1.7281]],
       grad_fn=<MulBackward0>)


那么，到这一步，transformer已经完成了推理，并且已经得到最终的输出结果。

接下来从sentence_transformer和vllm的源码分别处理后续过程，一般包括池化层和正则层

# sentence_transformer
## 池化层操作
Qwen3配置的是pooling_mode_lasttoken策略，也就是改进的最后一层的输出作为最终的输出。

In [21]:
attention_mask

tensor([1, 1, 1, 1, 1, 1, 1])

In [22]:
seq_len, hidden_dim = hidden_states.shape  # 只有2个维度

# 找到最后一个有效token的位置
# attention_mask 例如: [1, 1, 1, 0, 0]
last_token_index = attention_mask.nonzero()[-1].item()  # 直接找最后一个1的索引
# 或者: last_token_index = (attention_mask == 1).nonzero()[-1].item()

# 提取该位置的embedding
embedding = hidden_states[last_token_index]  # 直接索引，形状 [hidden_dim]
print(embedding)

tensor([-0.6450, -5.1673, -0.9345,  ..., -1.2473, -1.2305, -1.7281],
       grad_fn=<SelectBackward0>)


In [46]:
# 正则化操作
embedding_result = F.normalize(embedding, p=2, dim=0)
local_embed = embedding_result.detach().cpu().numpy()
print(local_embed[:10].tolist())

[-0.006285255774855614, -0.05035041272640228, -0.009106110781431198, -0.04767336696386337, -0.0066940803080797195, -0.023354386910796165, -0.05004096031188965, 0.002553714206442237, -0.14058202505111694, -0.008917026221752167]


In [47]:
import numpy as np
# 定义余弦相似度算法来对比和框架代码比较结果
def cosine_similarity(a, b):
    """
    计算两个向量的余弦相似度
    """
    # 计算两个向量的模
    a_norm = np.linalg.norm(a)
    b_norm = np.linalg.norm(b)

    # 计算两个向量的点积
    dot_product = np.dot(a, b)

    # 计算余弦相似度
    similarity = dot_product / (a_norm * b_norm)
    return similarity


In [48]:
# 对比和本地vllm部署的接口结果
import requests

url = "http://127.0.0.1:8000/v1/embeddings"
payload = {
    "model": "Qwen/Qwen3-Embedding-0.6B",
    "input": "Hello Word, a test sentence"
}

resp = requests.post(url, json=payload)
vllm_embed = resp.json()['data'][0]['embedding']
vllm_embed_tensor = torch.tensor(vllm_embed)
vllm_embed_tensor

tensor([-0.0063, -0.0502, -0.0091,  ..., -0.0122, -0.0120, -0.0168])

In [49]:
print(
    cosine_similarity(local_embed, vllm_embed)
)

0.9999979375673835
